In [ ]:
import os
import math
import pickle
import numpy as np
import pandas as pd
import rustworkx as rx
import geopandas as gpd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [5]:
# Load tabular data
school_data = pd.read_csv("data/ofsted_bradford_data.csv")
urn_name_map = school_data.set_index("URN")["School name"].to_dict()

In [6]:
# Load geospatial data
oa21lsoa = pd.read_excel("data/OA11_LAD21_LSOA11_MSOA11_LEP21_EN_v3.xlsx")
oa_gdf = gpd.read_file("data/Output_Areas_Dec_2011_Boundaries_EW_BGC_2022_-9162807391755440229.geojson")

selected_oas = oa21lsoa.loc[oa21lsoa["LAD21NM"].isin(["Bradford", "Leeds", "Calderdale", "Kirklees", "Craven", "Harrogate", "Pendle"]), "OA11CD"].unique()
bradford_gdf = oa_gdf.loc[oa_gdf["OA11CD"].isin(selected_oas)]

oa_unique = bradford_gdf.drop_duplicates(subset=["OA11CD"])
centroids_4326 = oa_unique.to_crs(epsg=27700).geometry.centroid.to_crs(epsg=4326)
oa_centroid_dict = dict(zip(oa_unique["OA11CD"], centroids_4326))

In [10]:
# Load graph data
with open("transit_graph_.pkl", "rb") as f:
    G = pickle.load(f)

with open("node_to_idx_.pkl", "rb") as f:
    node_to_idx = pickle.load(f)

In [11]:
# Check for invalid edge weights
invalid_edges = []
for edge_idx, edge_data in enumerate(G.edges()):
    if not isinstance(edge_data, dict) or not isinstance(edge_data.get("weight"), (int, float)):
        invalid_edges.append((edge_idx, edge_data))

if invalid_edges:
    print(f"Found {len(invalid_edges)} corrupted edges. Example: {invalid_edges[0]}")

In [12]:
# Algorithmic Optimization: Reverse the graph for backward single-source Dijkstra
# This allows calculating paths from ALL output areas to a school in one pass.
# G.reverse()

In [13]:
# Global state for worker processes
def process_school(school_dict):
    school_name = school_dict["Name"]
    school_urn = school_dict["URN"]
    
    # Destination index on original graph is the starting index on reversed graph
    dest_idx = node_to_idx.get(school_urn)
    if dest_idx is None:
        return None 

    def get_weight(edge):
       return float(edge.get("weight"))
    
    # Execute Dijkstra once per school to find paths and lengths from all nodes
    # Delegating iteration entirely to the Rust backend
    lengths = rx.dijkstra_shortest_path_lengths(G, dest_idx, get_weight)
    paths = rx.dijkstra_shortest_paths(G, dest_idx, None, get_weight)

    school_records = []
    
    for oa in selected_oas:
        oa_idx = node_to_idx.get(oa)
            
        # The path returned is School -> OA (on reversed graph). Reverse for OA -> School.
        path_indices = paths[oa_idx][::-1]
        optimal_path = [G.get_node_data(i)["name"] for i in path_indices]
        optimal_duration = lengths[oa_idx]
        
        oa_cent = oa_centroid_dict[oa]
        school_records.append({
            "School_Name": school_name,
            "School_Lat": school_dict["Lat"],
            "School_Lon": school_dict["Lon"],
            "OA_Code": oa,
            "OA_Centroid_Y": oa_cent.y,
            "OA_Centroid_X": oa_cent.x,
            "Duration_Seconds": optimal_duration,
            "Path": optimal_path
        })
        
    return pd.DataFrame(school_records)

In [14]:
OUTPUT_CSV = "school_oa_bus_durations_testing.csv"

file_exists = os.path.exists(OUTPUT_CSV)
completed_schools = set(pd.read_csv(OUTPUT_CSV)["School_Name"]) if file_exists else {}

In [15]:
# Filter out schools that are already completed before processing
pending_schools = [s for s in school_data.to_dict('records') if s["Name"] not in completed_schools]

print(f"Skipping {len(school_data) - len(pending_schools)} completed schools.")
print(f"Processing {len(pending_schools)} schools sequentially...")

# Sequential processing loop
for school in tqdm(pending_schools, desc="Calculating Routes"):
    df_result = process_school(school)
    
    if df_result is not None and not df_result.empty:
        # Append results to CSV
        df_result.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
        file_exists = True

Skipping 0 completed schools.
Processing 104 schools sequentially...


Calculating Routes:   0%|          | 0/104 [00:00<?, ?it/s]


IndexError: No node found for index

In [ ]:
# Filter out schools that are already completed before dispatching jobs
pending_schools = [s for s in school_data.to_dict('records') if s["Name"] not in completed_schools]

print(f"Skipping {len(school_data) - len(pending_schools)} completed schools.")
print(f"Processing {len(pending_schools)} schools across {os.cpu_count()} CPU cores...")

# Process parallelization
with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    futures = {executor.submit(process_school, school): school["Name"] for school in pending_schools}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Calculating Routes"):
        school_name = futures[future]
        df_result = future.result()

        # Write to file
        df_result.to_csv(OUTPUT_CSV, mode='a', index=False, header=not file_exists)
        file_exists = True

Skipping 0 completed schools.
Processing 100 schools across 22 CPU cores...


Calculating Routes:   0%|          | 0/100 [00:00<?, ?it/s]


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.